# T3.1  RO-Crate Metadata Generation

This notebook creates the `ro-crate-metadata.json` file for the full experiment package.

The generated RO-Crate describes the input datasets, source DOI, code, notebooks, trained models, output data, authors, licences, external repositories, and related persistent identifiers.

In [13]:
from pathlib import Path
import json
import subprocess
import sys

# Detect repository root whether notebook is run from /notebooks or root
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

ROCRATE_PATH = ROOT / "ro-crate-metadata.json"
VALIDATION_DIR = ROOT / "docs" / "validation"
VALIDATION_OUTPUT = VALIDATION_DIR / "ro-crate-validation.txt"

print("Repository root:", ROOT)
print("RO-Crate output:", ROCRATE_PATH)

Repository root: C:\Users\Saad\Downloads\TU WIEN 26 Summer\DS UE\# part 3\DataStewardshipSS2026
RO-Crate output: C:\Users\Saad\Downloads\TU WIEN 26 Summer\DS UE\# part 3\DataStewardshipSS2026\ro-crate-metadata.json


## Project identifiers and known external resources

This cell stores verified project-level identifiers and links. These values should be updated here if a DOI, URL, or repository link changes later.

In [14]:
PROJECT_TITLE = "Prediction of Heavy Metal Concentrations (Pb, Cd) in Precipitation Using Machine Learning"

SOURCE_DATASET_DOI = "https://doi.org/10.48436/b0g4h-rv840"
SOURCE_DATASET_LICENSE = "https://creativecommons.org/licenses/by-nc-sa/4.0/"

GITHUB_REPOSITORY = "https://github.com/fambrogi/DataStewardshipSS2026"
DBREPO_ENTRY = "https://test.dbrepo.tuwien.ac.at/database/bfa4385b-54a9-4ae3-b4f4-cb503d7bb016/info"
ZENODO_DOI = "https://doi.org/10.5281/zenodo.20423764"

TUWRD_MODEL_DEPOSIT_DOI = "https://doi.org/10.70124/rwg1b-kfb59"
TUWRD_GENERATED_DATA_DEPOSIT_DOI = "https://doi.org/10.70124/kcmvz-fkw96"

## Creators

The four project members are described with names, roles, and ORCID identifiers.

In [15]:
creators = [
    {
        "@id": "#saad-rashidul-amin",
        "@type": "Person",
        "name": "Saad Rashidul Amin",
        "identifier": "https://orcid.org/0009-0004-0529-5546",
        "roleName": "Role A"
    },
    {
        "@id": "#puthenpurayil-biju-vijayalakshmi",
        "@type": "Person",
        "name": "Puthenpurayil Biju Vijayalakshmi",
        "identifier": "https://orcid.org/0009-0000-1739-2336",
        "roleName": "Role B"
    },
    {
        "@id": "#federico-ambrogi",
        "@type": "Person",
        "name": "Federico Ambrogi",
        "identifier": "https://orcid.org/0000-0002-9486-0444",
        "roleName": "Role C"
    },
    {
        "@id": "#farooq-mian-azan",
        "@type": "Person",
        "name": "Farooq Mian Azan",
        "identifier": "https://orcid.org/0009-0006-7973-2483",
        "roleName": "Role D"
    }
]

## Repository artefacts

This list contains the local files and folders that should be represented in the RO-Crate. The notebook checks whether each path exists before including it, so renamed or missing files are easier to detect.

In [24]:
artefacts = [
    "README.md",
    "LICENSE",
    "data/precipitationdata.csv",
    "data/stationcoordinates.csv",
    "docs/model-card.md",
    "docs/provenance.md",
    "docs/data_dictionary.md",
    "sql/create_tables.sql",
    "notebooks/predict_heavymetal_precipitation.ipynb",
    "notebooks/T2_1_dbrepo_schema_creation.ipynb",
    "notebooks/T3_1_ROCrate.ipynb",
    "notebooks/T3_3-FAIRML.ipynb",
    "notebooks/T3_5_ModelCard.ipynb",
    "src/models.py",
    "src/preprocessing.py",
    "src/evaluation.py",
    "src/utils.py",
    "outputs/models/model_cd_randomforest.pkl",
    "outputs/models/model_multi_randomforest.pkl",
    "outputs/metadata/precipitationdata_croissant.json",
    "outputs/metadata/stationcoordinates_croissant.json",
    "outputs/metadata/FAIRML_model_cd",
    "outputs/metadata/FAIRML_model_multi",
    "outputs/figures/"
]

existing_artefacts = []
missing_artefacts = []

for path in artefacts:
    if (ROOT / path).exists():
        existing_artefacts.append(path)
    else:
        missing_artefacts.append(path)

print("Existing artefacts:")
for item in existing_artefacts:
    print("✓", item)

print("\nMissing artefacts:")
for item in missing_artefacts:
    print("✗", item)

Existing artefacts:
✓ README.md
✓ LICENSE
✓ data/precipitationdata.csv
✓ data/stationcoordinates.csv
✓ docs/model-card.md
✓ docs/provenance.md
✓ docs/data_dictionary.md
✓ sql/create_tables.sql
✓ notebooks/predict_heavymetal_precipitation.ipynb
✓ notebooks/T2_1_dbrepo_schema_creation.ipynb
✓ notebooks/T3_1_ROCrate.ipynb
✓ notebooks/T3_3-FAIRML.ipynb
✓ notebooks/T3_5_ModelCard.ipynb
✓ src/models.py
✓ src/preprocessing.py
✓ src/evaluation.py
✓ src/utils.py
✓ outputs/models/model_cd_randomforest.pkl
✓ outputs/models/model_multi_randomforest.pkl
✓ outputs/metadata/precipitationdata_croissant.json
✓ outputs/metadata/stationcoordinates_croissant.json
✓ outputs/metadata/FAIRML_model_cd
✓ outputs/metadata/FAIRML_model_multi
✓ outputs/figures/

Missing artefacts:


## Build RO-Crate graph

This cell constructs the RO-Crate JSON-LD graph. External DOI and repository records are included even though they are not local files.

In [25]:
def file_entity(path, name, encoding_format=None, entity_type="File", extra=None):
    entity = {
        "@id": path,
        "@type": entity_type,
        "name": name
    }
    if encoding_format:
        entity["encodingFormat"] = encoding_format
    if extra:
        entity.update(extra)
    return entity


has_part = [{"@id": path} for path in existing_artefacts]

# External entities must also be linked from the root using hasPart/isBasedOn/subjectOf.
# To satisfy strict RO-Crate validation, we include the main external deposits in hasPart as well.
has_part.extend([
    {"@id": SOURCE_DATASET_DOI},
    {"@id": DBREPO_ENTRY},
    {"@id": TUWRD_MODEL_DEPOSIT_DOI},
    {"@id": TUWRD_GENERATED_DATA_DEPOSIT_DOI}
])

graph = [
    {
        "@id": "ro-crate-metadata.json",
        "@type": "CreativeWork",
        "about": {"@id": "./"},
        "conformsTo": {"@id": "https://w3id.org/ro/crate/1.1"}
    },
    {
        "@id": "./",
        "@type": "Dataset",
        "name": PROJECT_TITLE,
        "description": "FAIR Data Science group project for predicting Pb and Cd concentrations in Austrian precipitation samples using machine learning and environmental precipitation chemistry data.",
        "datePublished": "2026-05-28",
        "creator": [{"@id": creator["@id"]} for creator in creators],
        "license": {"@id": "LICENSE"},
        "hasPart": has_part,
        "isBasedOn": {"@id": SOURCE_DATASET_DOI},
        "subjectOf": [
            {"@id": GITHUB_REPOSITORY},
            {"@id": ZENODO_DOI}
        ]
    }
]

graph.extend(creators)

graph.extend([
    file_entity(
        "README.md",
        "Project README",
        "text/markdown"
    ),

    file_entity(
        "LICENSE",
        "MIT License",
        entity_type="CreativeWork",
        extra={
            "description": "MIT licence for the software code and trained model artefacts in this project."
        }
    ),

    file_entity(
        "data/precipitationdata.csv",
        "Precipitation chemistry data",
        "text/csv",
        extra={
            "isBasedOn": {"@id": SOURCE_DATASET_DOI},
            "license": {"@id": SOURCE_DATASET_LICENSE}
        }
    ),

    file_entity(
        "data/stationcoordinates.csv",
        "Station coordinates data",
        "text/csv",
        extra={
            "isBasedOn": {"@id": SOURCE_DATASET_DOI},
            "license": {"@id": SOURCE_DATASET_LICENSE}
        }
    ),

    file_entity(
        "outputs/metadata/precipitationdata_croissant.json",
        "Croissant metadata for precipitationdata.csv",
        "application/ld+json",
        extra={"about": {"@id": "data/precipitationdata.csv"}}
    ),

    file_entity(
        "outputs/metadata/stationcoordinates_croissant.json",
        "Croissant metadata for stationcoordinates.csv",
        "application/ld+json",
        extra={"about": {"@id": "data/stationcoordinates.csv"}}
    ),

    file_entity(
        "outputs/metadata/FAIRML_model_cd",
        "FAIR4ML metadata for Cd Random Forest model",
        "application/ld+json",
        extra={"about": {"@id": "outputs/models/model_cd_randomforest.pkl"}}
    ),

    file_entity(
        "outputs/metadata/FAIRML_model_multi",
        "FAIR4ML metadata for multi-output Random Forest model",
        "application/ld+json",
        extra={"about": {"@id": "outputs/models/model_multi_randomforest.pkl"}}
    ),

    file_entity(
        "docs/model-card.md",
        "Model Card",
        "text/markdown",
        extra={
            "about": [
                {"@id": "outputs/models/model_cd_randomforest.pkl"},
                {"@id": "outputs/models/model_multi_randomforest.pkl"}
            ]
        }
    ),

    file_entity("docs/provenance.md", "Provenance documentation", "text/markdown"),
    file_entity("docs/data_dictionary.md", "Data dictionary", "text/markdown"),
    file_entity("sql/create_tables.sql", "SQL schema definition", "application/sql"),

    file_entity(
        "notebooks/predict_heavymetal_precipitation.ipynb",
        "Main machine learning workflow notebook",
        "application/x-ipynb+json",
        "SoftwareSourceCode",
        {"programmingLanguage": "Python"}
    ),

    file_entity(
        "notebooks/T2_1_dbrepo_schema_creation.ipynb",
        "DBRepo schema creation notebook",
        "application/x-ipynb+json",
        "SoftwareSourceCode",
        {"programmingLanguage": "Python"}
    ),

    file_entity(
        "notebooks/T3_1_ROCrate.ipynb",
        "RO-Crate metadata generation notebook",
        "application/x-ipynb+json",
        "SoftwareSourceCode",
        {"programmingLanguage": "Python"}
    ),

    file_entity(
        "notebooks/T3_3-FAIRML.ipynb",
        "FAIR4ML metadata generation notebook",
        "application/x-ipynb+json",
        "SoftwareSourceCode",
        {"programmingLanguage": "Python"}
    ),
    
    file_entity(
        "notebooks/T3_5_ModelCard.ipynb",
        "Model Card generation notebook",
        "application/x-ipynb+json",
        "SoftwareSourceCode",
        {"programmingLanguage": "Python"}
    ),

    file_entity(
        "src/models.py",
        "Model training functions",
        entity_type="SoftwareSourceCode",
        extra={"programmingLanguage": "Python", "license": {"@id": "LICENSE"}}
    ),

    file_entity(
        "src/preprocessing.py",
        "Preprocessing functions",
        entity_type="SoftwareSourceCode",
        extra={"programmingLanguage": "Python", "license": {"@id": "LICENSE"}}
    ),

    file_entity(
        "src/evaluation.py",
        "Evaluation and plotting functions",
        entity_type="SoftwareSourceCode",
        extra={"programmingLanguage": "Python", "license": {"@id": "LICENSE"}}
    ),

    file_entity(
        "src/utils.py",
        "Utility functions",
        entity_type="SoftwareSourceCode",
        extra={"programmingLanguage": "Python", "license": {"@id": "LICENSE"}}
    ),

    file_entity(
        "outputs/models/model_cd_randomforest.pkl",
        "Cadmium Random Forest model",
        "application/octet-stream",
        extra={
            "description": "Trained Random Forest model artifact for Cd prediction in the sequential workflow.",
            "license": {"@id": "LICENSE"},
            "sameAs": {"@id": TUWRD_MODEL_DEPOSIT_DOI}
        }
    ),

    file_entity(
        "outputs/models/model_multi_randomforest.pkl",
        "Multi-output Random Forest model",
        "application/octet-stream",
        extra={
            "description": "Trained Random Forest model artifact for simultaneous Pb and Cd prediction.",
            "license": {"@id": "LICENSE"},
            "sameAs": {"@id": TUWRD_MODEL_DEPOSIT_DOI}
        }
    ),

    {
        "@id": "outputs/figures/",
        "@type": "Dataset",
        "name": "Generated figures and evaluation visualizations",
        "sameAs": {"@id": TUWRD_GENERATED_DATA_DEPOSIT_DOI}
    },

    {
        "@id": SOURCE_DATASET_DOI,
        "@type": "Dataset",
        "name": "Concentrations of major ions in wet precipitation samples in Austria",
        "license": {"@id": SOURCE_DATASET_LICENSE},
        "creator": [
            {"@id": "#peter-redl"},
            {"@id": "#thomas-steinkogler"},
            {"@id": "#anne-kasper-giebl"}
        ]
    },

    {
        "@id": "#peter-redl",
        "@type": "Person",
        "name": "Peter Redl"
    },

    {
        "@id": "#thomas-steinkogler",
        "@type": "Person",
        "name": "Thomas Steinkogler"
    },

    {
        "@id": "#anne-kasper-giebl",
        "@type": "Person",
        "name": "Anne Kasper-Giebl"
    },

    {
        "@id": GITHUB_REPOSITORY,
        "@type": "SoftwareSourceCode",
        "name": "GitHub repository",
        "license": {"@id": "LICENSE"}
    },

    {
        "@id": DBREPO_ENTRY,
        "@type": "Dataset",
        "name": "DBRepo database: austria_precipitation_project"
    },

    {
        "@id": ZENODO_DOI,
        "@type": "CreativeWork",
        "name": "Zenodo archived GitHub release"
    },

    {
        "@id": TUWRD_MODEL_DEPOSIT_DOI,
        "@type": "CreativeWork",
        "name": "TUWRD model deposit",
        "license": {"@id": "LICENSE"}
    },

    {
        "@id": TUWRD_GENERATED_DATA_DEPOSIT_DOI,
        "@type": "Dataset",
        "name": "TUWRD generated data deposit"
    }
])

ro_crate = {
    "@context": "https://w3id.org/ro/crate/1.1/context",
    "@graph": graph
}

## Write RO-Crate metadata file

This cell writes the generated RO-Crate JSON-LD file to the repository root as `ro-crate-metadata.json`.

In [26]:
with open(ROCRATE_PATH, "w", encoding="utf-8") as f:
    json.dump(ro_crate, f, indent=2, ensure_ascii=False)

print(f"RO-Crate written to: {ROCRATE_PATH}")

RO-Crate written to: C:\Users\Saad\Downloads\TU WIEN 26 Summer\DS UE\# part 3\DataStewardshipSS2026\ro-crate-metadata.json


## Validate JSON syntax

This cell checks whether the generated `ro-crate-metadata.json` is valid JSON before running the RO-Crate validator.

In [27]:
with open(ROCRATE_PATH, "r", encoding="utf-8") as f:
    loaded = json.load(f)

print("JSON syntax valid.")
print("Number of graph entities:", len(loaded["@graph"]))

JSON syntax valid.
Number of graph entities: 39


## Validate RO-Crate

This cell runs `rocrate-validator` from the repository root and stores the validation output under `docs/validation/ro-crate-validation.txt`.

In [20]:
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

try:
    result = subprocess.run(
        ["rocrate-validator", str(ROOT)],
        cwd=ROOT,
        text=True,
        capture_output=True
    )
except FileNotFoundError:
    print("rocrate-validator is not installed. Install it first using:")
    print("pip install rocrate-validator")
else:
    output = result.stdout + "\n" + result.stderr
    VALIDATION_OUTPUT.write_text(output, encoding="utf-8")

    print(output)
    print(f"\nValidation output saved to: {VALIDATION_OUTPUT}")
    print("Return code:", result.returncode)

rocrate-validator is not installed. Install it first using:
pip install rocrate-validator
